# Customer Retention & Churn Analysis
**Dataset:** IBM Telco Customer Churn  
**Tool:** Python (pandas)  
**Task:** Future Interns — Data Science & Analytics Task 2 (2026)

Run each cell top to bottom using **Shift + Enter**.

!pip install pandas numpy

---
## Step 1 — Load the Dataset

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset — file is in the same folder as this notebook
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Dataset loaded successfully!')
print('Shape:', df.shape)
df.head()

Dataset loaded successfully!
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


---
## Step 2 — Inspect the Raw Data

In [2]:
print('Columns:', df.columns.tolist())

Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [3]:
# Data types — look for numeric columns stored as 'object'
print(df.dtypes)

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [4]:
# Missing values
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [5]:
# Unique values in key categorical columns
for col in ['Churn', 'Contract', 'InternetService', 'PaymentMethod']:
    print(f'{col}: {df[col].unique()}')

Churn: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Contract: <StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str
InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
PaymentMethod: <StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str


In [6]:
# Numeric ranges
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [7]:
# Churn distribution
print(df['Churn'].value_counts())
print()
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


---
## Step 3 — Clean the Data

In [8]:
# Fix TotalCharges — stored as text; blank rows become NaN, filled with MonthlyCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])
print('TotalCharges fixed. Missing remaining:', df['TotalCharges'].isnull().sum())

TotalCharges fixed. Missing remaining: 0


In [9]:
# ChurnFlag — convert Yes/No to 1/0 for arithmetic
df['ChurnFlag'] = (df['Churn'] == 'Yes').astype(int)
print('ChurnFlag created:')
print(df['ChurnFlag'].value_counts())

ChurnFlag created:
ChurnFlag
0    5174
1    1869
Name: count, dtype: int64


In [10]:
# TenureBand — group 0-72 months into 6 cohort bands
bins   = [0, 12, 24, 36, 48, 60, 72]
labels = ['0-12 mo', '13-24 mo', '25-36 mo', '37-48 mo', '49-60 mo', '61-72 mo']
df['TenureBand'] = pd.cut(df['tenure'], bins=bins, labels=labels, include_lowest=True)
print('TenureBand created:')
print(df['TenureBand'].value_counts().sort_index())

TenureBand created:
TenureBand
0-12 mo     2186
13-24 mo    1024
25-36 mo     832
37-48 mo     762
49-60 mo     832
61-72 mo    1407
Name: count, dtype: int64


In [11]:
# SeniorCitizenLabel — make 0/1 human-readable
df['SeniorCitizenLabel'] = df['SeniorCitizen'].map({0: 'Non-Senior', 1: 'Senior'})

# Save cleaned dataset
df.to_csv('telco_cleaned.csv', index=False)
print('Clean dataset saved! Shape:', df.shape)

Clean dataset saved! Shape: (7043, 24)


---
## Step 4 — Calculate KPIs

In [12]:
total    = len(df)
churned  = df['ChurnFlag'].sum()
retained = total - churned

churn_rate              = round(churned / total * 100, 2)
retention_rate          = round(retained / total * 100, 2)
avg_monthly             = round(df['MonthlyCharges'].mean(), 2)
monthly_revenue_at_risk = round(df[df['ChurnFlag']==1]['MonthlyCharges'].sum(), 2)
avg_tenure_churned      = round(df[df['ChurnFlag']==1]['tenure'].mean(), 1)
avg_tenure_retained     = round(df[df['ChurnFlag']==0]['tenure'].mean(), 1)
total_revenue_lost      = round(df[df['ChurnFlag']==1]['TotalCharges'].sum(), 2)

kpi_df = pd.DataFrame({
    'Metric': [
        'Total Customers', 'Churned Customers', 'Retained Customers',
        'Churn Rate (%)', 'Retention Rate (%)', 'Avg Monthly Charge ($)',
        'Monthly Revenue at Risk ($)', 'Avg Tenure - Churned (months)',
        'Avg Tenure - Retained (months)', 'Total Revenue Lost ($)'
    ],
    'Value': [
        total, int(churned), int(retained), churn_rate, retention_rate,
        avg_monthly, monthly_revenue_at_risk, avg_tenure_churned,
        avg_tenure_retained, total_revenue_lost
    ]
})

kpi_df.to_csv('01_kpi_summary.csv', index=False)
print('KPIs saved!')
kpi_df

KPIs saved!


,Metric,Value
0,Total Customers,7043.00
1,Churned Customers,1869.00
2,Retained Customers,5174.00
3,Churn Rate (%),26.54
4,Retention Rate (%),73.46
5,Avg Monthly Charge ($),64.76
6,Monthly Revenue at Risk ($),139130.85
7,Avg Tenure - Churned (months),18.00
8,Avg Tenure - Retained (months),37.60
9,Total Revenue Lost ($),2862926.90


---
## Step 5 — Build Aggregated Tables for Power BI

In [13]:
# Universal helper — call this for any segment column
def churn_by_segment(df, segment_col, filename):
    result = df.groupby(segment_col, observed=True).agg(
        Total   = ('ChurnFlag', 'count'),
        Churned = ('ChurnFlag', 'sum')
    ).reset_index()
    result['Retained']      = result['Total'] - result['Churned']
    result['ChurnRate']     = (result['Churned'] / result['Total'] * 100).round(2)
    result['RetentionRate'] = (100 - result['ChurnRate']).round(2)
    result.to_csv(filename, index=False)
    print(f'Saved {filename}')
    return result

print('Helper function ready.')

Helper function ready.


In [21]:
contract = churn_by_segment(df, 'Contract', '02_churn_by_contract.csv')
contract

Saved 02_churn_by_contract.csv


,Contract,Total,Churned,Retained,ChurnRate,RetentionRate
0,Month-to-month,3875,1655,2220,42.71,57.29
1,One year,1473,166,1307,11.27,88.73
2,Two year,1695,48,1647,2.83,97.17


In [14]:
tenure_band = churn_by_segment(df, 'TenureBand', '03_churn_by_tenure_band.csv')
tenure_band

Saved 03_churn_by_tenure_band.csv


,TenureBand,Total,Churned,Retained,ChurnRate,RetentionRate
0,0-12 mo,2186,1037,1149,47.44,52.56
1,13-24 mo,1024,294,730,28.71,71.29
2,25-36 mo,832,180,652,21.63,78.37
3,37-48 mo,762,145,617,19.03,80.97
4,49-60 mo,832,120,712,14.42,85.58
5,61-72 mo,1407,93,1314,6.61,93.39


In [15]:
internet = churn_by_segment(df, 'InternetService', '04_churn_by_internet.csv')
internet

Saved 04_churn_by_internet.csv


,InternetService,Total,Churned,Retained,ChurnRate,RetentionRate
0,DSL,2421,459,1962,18.96,81.04
1,Fiber optic,3096,1297,1799,41.89,58.11
2,No,1526,113,1413,7.40,92.60


In [16]:
payment = churn_by_segment(df, 'PaymentMethod', '05_churn_by_payment.csv')
payment

Saved 05_churn_by_payment.csv


,PaymentMethod,Total,Churned,Retained,ChurnRate,RetentionRate
0,Bank transfer (automatic),1544,258,1286,16.71,83.29
1,Credit card (automatic),1522,232,1290,15.24,84.76
2,Electronic check,2365,1071,1294,45.29,54.71
3,Mailed check,1612,308,1304,19.11,80.89


In [17]:
senior = churn_by_segment(df, 'SeniorCitizenLabel', '06_churn_by_senior.csv')
senior

Saved 06_churn_by_senior.csv


,SeniorCitizenLabel,Total,Churned,Retained,ChurnRate,RetentionRate
0,Non-Senior,5901,1393,4508,23.61,76.39
1,Senior,1142,476,666,41.68,58.32


In [19]:
gender = churn_by_segment(df, 'gender', '07_churn_by_gender.csv')
gender

Saved 07_churn_by_gender.csv


,gender,Total,Churned,Retained,ChurnRate,RetentionRate
0,Female,3488,939,2549,26.92,73.08
1,Male,3555,930,2625,26.16,73.84


In [22]:
# Service add-on impact — WITH vs WITHOUT each feature
services = ['OnlineSecurity', 'TechSupport', 'OnlineBackup',
            'DeviceProtection', 'StreamingTV', 'StreamingMovies']
rows = []
for svc in services:
    subset = df[df[svc].isin(['Yes', 'No'])]
    for val in ['Yes', 'No']:
        grp = subset[subset[svc] == val]
        rows.append({
            'Service'   : svc,
            'Subscribed': val,
            'Total'     : len(grp),
            'Churned'   : int(grp['ChurnFlag'].sum()),
            'ChurnRate' : round(grp['ChurnFlag'].mean() * 100, 2)
        })
addons = pd.DataFrame(rows)
addons.to_csv('08_churn_by_service_addon.csv', index=False)
print('Saved 08_churn_by_service_addon.csv')
addons

Saved 08_churn_by_service_addon.csv


,Service,Subscribed,Total,Churned,ChurnRate
0,OnlineSecurity,Yes,2019,295,14.61
1,OnlineSecurity,No,3498,1461,41.77
2,TechSupport,Yes,2044,310,15.17
3,TechSupport,No,3473,1446,41.64
4,OnlineBackup,Yes,2429,523,21.53
5,OnlineBackup,No,3088,1233,39.93
6,DeviceProtection,Yes,2422,545,22.50
7,DeviceProtection,No,3095,1211,39.13
8,StreamingTV,Yes,2707,814,30.07
9,StreamingTV,No,2810,942,33.52


In [23]:
# Cohort retention matrix — Tenure Band x Contract
cohort = df.groupby(['TenureBand', 'Contract'], observed=True).agg(
    Total   = ('ChurnFlag', 'count'),
    Churned = ('ChurnFlag', 'sum')
).reset_index()
cohort['RetentionRate'] = ((cohort['Total'] - cohort['Churned']) / cohort['Total'] * 100).round(2)
cohort['ChurnRate']     = (cohort['Churned'] / cohort['Total'] * 100).round(2)
cohort.to_csv('09_cohort_retention.csv', index=False)
print('Saved 09_cohort_retention.csv')
cohort

Saved 09_cohort_retention.csv


,TenureBand,Contract,Total,Churned,RetentionRate,ChurnRate
0,0-12 mo,Month-to-month,1994,1024,48.65,51.35
1,0-12 mo,One year,124,13,89.52,10.48
2,0-12 mo,Two year,68,0,100.00,0.00
3,13-24 mo,Month-to-month,737,278,62.28,37.72
4,13-24 mo,One year,197,16,91.88,8.12
5,13-24 mo,Two year,90,0,100.00,0.00
6,25-36 mo,Month-to-month,486,158,67.49,32.51
7,25-36 mo,One year,250,20,92.00,8.00
8,25-36 mo,Two year,96,2,97.92,2.08
9,37-48 mo,Month-to-month,316,106,66.46,33.54


In [24]:
# Customer Lifetime Value
clv = df.groupby(['Contract', 'Churn']).agg(
    AvgTenure  = ('tenure',         'mean'),
    AvgMonthly = ('MonthlyCharges', 'mean'),
    AvgTotal   = ('TotalCharges',   'mean'),
    Count      = ('customerID',     'count')
).reset_index().round(2)
clv.to_csv('10_customer_lifetime_value.csv', index=False)
print('Saved 10_customer_lifetime_value.csv')
clv

Saved 10_customer_lifetime_value.csv


,Contract,Churn,AvgTenure,AvgMonthly,AvgTotal,Count
0,Month-to-month,No,21.03,61.46,1521.93,2220
1,Month-to-month,Yes,14.02,73.02,1164.46,1655
2,One year,No,41.67,62.51,2901.36,1307
3,One year,Yes,44.96,85.05,4066.21,166
4,Two year,No,56.60,60.01,3656.91,1647
5,Two year,Yes,61.27,86.78,5432.36,48


---
## Step 6 — Summary of Key Insights

In [25]:
print('=' * 55)
print('  CUSTOMER CHURN ANALYSIS - KEY FINDINGS')
print('=' * 55)
print(f'  Total Customers          : {total:,}')
print(f'  Churned                  : {int(churned):,} ({churn_rate}%)')
print(f'  Retained                 : {int(retained):,} ({retention_rate}%)')
print(f'  Monthly Revenue at Risk  : ${monthly_revenue_at_risk:,.2f}')
print(f'  Total Revenue Lost       : ${total_revenue_lost:,.2f}')
print(f'  Avg Tenure - Churned     : {avg_tenure_churned} months')
print(f'  Avg Tenure - Retained    : {avg_tenure_retained} months')
print()
print('  Churn rate by contract type:')
for _, row in contract.iterrows():
    print(f'    {row["Contract"]:<22} {row["ChurnRate"]}%')
print()
print('  Churn rate by internet service:')
for _, row in internet.iterrows():
    print(f'    {row["InternetService"]:<22} {row["ChurnRate"]}%')
print()
print('  Churn rate by payment method (highest first):')
for _, row in payment.sort_values('ChurnRate', ascending=False).iterrows():
    print(f'    {row["PaymentMethod"]:<35} {row["ChurnRate"]}%')
print('=' * 55)
print('  All CSV files saved - ready to import into Power BI!')
print('=' * 55)

  CUSTOMER CHURN ANALYSIS - KEY FINDINGS
  Total Customers          : 7,043
  Churned                  : 1,869 (26.54%)
  Retained                 : 5,174 (73.46%)
  Monthly Revenue at Risk  : $139,130.85
  Total Revenue Lost       : $2,862,926.90
  Avg Tenure - Churned     : 18.0 months
  Avg Tenure - Retained    : 37.6 months

  Churn rate by contract type:
    Month-to-month         42.71%
    One year               11.27%
    Two year               2.83%

  Churn rate by internet service:
    DSL                    18.96%
    Fiber optic            41.89%
    No                     7.4%

  Churn rate by payment method (highest first):
    Electronic check                    45.29%
    Mailed check                        19.11%
    Bank transfer (automatic)           16.71%
    Credit card (automatic)             15.24%
  All CSV files saved - ready to import into Power BI!
